In [13]:
import pandas as pd
import numpy as np
import spacy
from textblob import TextBlob
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, classification_report, precision_recall_curve
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import chi2
from tqdm.auto import tqdm
import json

In [2]:
#Identify base directory to ensure portability
BASE_DIR = Path.cwd().resolve().parent
MODELS_DIR = BASE_DIR / "models"
DATA = BASE_DIR / "data/processed/distilled.csv"

In [3]:
df = pd.read_csv(DATA)
display(df)

,Unnamed: 0,text,propaganda
0,0,Outrage as Donald Trump suggests injecting dis...,"[{""span"":[0,6],""technique"":""Loaded_Language""}]"
1,1,The senator's vile betrayal of working familie...,"[{""span"":[14,18],""technique"":""Loaded_Language""..."
2,2,Brave freedom fighters resist the tyrannical o...,"[{""span"":[0,5],""technique"":""Loaded_Language""},..."
3,3,The corrupt elites are bleeding this country d...,"[{""span"":[4,10],""technique"":""Loaded_Language""}..."
4,4,A catastrophic failure of leadership has plung...,"[{""span"":[2,13],""technique"":""Loaded_Language""}..."
...,...,...,...
5852,5852,Altered Election Documents Tied To Florida Dem...,"[{'span': [86, 101], 'technique': 'Loaded_Lang..."
5853,5853,Migrant Caravan Reach Border & Climb Atop Fenc...,"[{'span': [31, 62], 'technique': 'Loaded_Langu..."
5854,5854,Guardian ups its vilification of Julian Assang...,"[{'span': [17, 29], 'technique': 'Loaded_Langu..."
5855,5855,This Guardian Fake News Story Proves That The ...,"[{'span': [0, 68], 'technique': 'Doubt'}, {'sp..."


In [4]:
#Now let's add simple, cheap features to the data
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

In [5]:
def extract_features(text):
    if not isinstance(text, str) or not text.strip():
        return [0] * 13

    blob = TextBlob(text)
    doc = nlp(text)
    tokens = [t for t in doc if not t.is_space]
    words = [t for t in tokens if not t.is_punct]
    word_cnt = max(len(words), 1)

    return [
        len(text), len(words),
        np.mean([len(w.text) for w in words]) if words else 0,
        sum(1 for w in words if w.text.isupper() and len(w.text) > 1) / word_cnt,
        sum(1 for t in tokens if any(c in '!?."' for c in t.text)) / word_cnt,
        len(doc), sum(1 for w in words if w.text.istitle()), len(tokens),
        blob.sentiment.polarity, blob.sentiment.subjectivity,
        sum(1 for t in doc if t.pos_ == "ADJ") / len(tokens),
        sum(1 for t in doc if t.pos_ == "ADV") / len(tokens),
        len(set([w.text.lower() for w in words])) / word_cnt
    ]

In [26]:
def get_tech_list(val):
    tech_map = {
        "Red_Herring": "Whataboutism_Straw_Men_Red_Herring",
        "Minimisation": "Exaggeration_Minimisation"
    }
    try:
        items = json.loads(val.replace("'", '"')) if isinstance(val, str) else val
        techs = [i['technique'] for i in items]
        return list(set([tech_map.get(t, t) for t in techs]))
    except:
        return []

df['tech_list'] = df['propaganda'].apply(get_tech_list)

In [27]:
#Process Features
feature_cols = ['char_count', 'word_count', 'avg_word_len', 'caps_ratio', 'punct_ratio',
                'sent_count', 'title_count', 'total_tokens', 'polarity',
                'subjectivity', 'adj_density', 'adv_density', 'ttr']

In [28]:
handcrafted_features = np.array(df['text'].apply(extract_features).tolist())
features_df = pd.DataFrame(handcrafted_features, columns=feature_cols, index=df.index)
df = pd.concat([df, features_df], axis=1)
df.head()

KeyboardInterrupt: 

In [18]:
#Vectorize and split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(train_df['text'].fillna(""))
X_test_tfidf = tfidf.transform(test_df['text'].fillna(""))

In [19]:
#Enhance with keywords
all_techs = sorted(list(set([t for s in train_df['propaganda'].apply(get_tech_list) for t in s])))
mlb = MultiLabelBinarizer(classes=all_techs)

train_has_p = train_df['propaganda'].apply(get_tech_list).str.len() > 0
y_train_spec_full = mlb.fit_transform(train_df['propaganda'].apply(get_tech_list))

In [20]:
keyword_feats_train = []
keyword_feats_test = []

for i, tech in enumerate(all_techs):
    _, pval = chi2(X_train_tfidf, y_train_spec_full[:, i])
    top_indices = np.argsort(pval)[:15]

    #Create binary feature: does text contain any of these top 15 words?
    keyword_feats_train.append(X_train_tfidf[:, top_indices].sum(axis=1) > 0)
    keyword_feats_test.append(X_test_tfidf[:, top_indices].sum(axis=1) > 0)

In [21]:
X_train_final = np.hstack([train_df[feature_cols].values, X_train_tfidf.toarray(), np.array(keyword_feats_train).T.reshape(len(train_df), -1)])
X_test_final = np.hstack([test_df[feature_cols].values, X_test_tfidf.toarray(), np.array(keyword_feats_test).T.reshape(len(test_df), -1)])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled = scaler.transform(X_test_final)

In [22]:
#Train lightweight version of scanner, or gatekeeper model
y_train_bin = train_has_p.astype(int)
y_test_bin = (test_df['propaganda'].apply(get_tech_list).str.len() > 0).astype(int)

gatekeeper = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
gatekeeper.fit(X_train_scaled, y_train_bin)
print(f"Gatekeeper F1: {f1_score(y_test_bin, gatekeeper.predict(X_test_scaled)):.4f}")

Gatekeeper F1: 0.9082


In [23]:
#Train lightweight version of classifier model
X_train_spec = X_train_scaled[train_has_p]
y_train_spec = y_train_spec_full[train_has_p]

test_has_p = y_test_bin == 1
X_test_spec = X_test_scaled[test_has_p]
y_test_spec = mlb.transform(test_df[test_has_p]['propaganda'].apply(get_tech_list))

spec_model = MultiOutputClassifier(LogisticRegression(class_weight='balanced', max_iter=2000))
spec_model.fit(X_train_spec, y_train_spec)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.A :term:`predict_proba` method will be exposed only if `estimator` implementsit.,LogisticRegre...max_iter=2000)
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary ` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights invers

In [24]:
#Optimize thresholds
probs = spec_model.predict_proba(X_test_spec)
y_pred_opt = np.zeros(y_test_spec.shape)

for i, tech in enumerate(all_techs):
    p, r, thresholds = precision_recall_curve(y_test_spec[:, i], probs[i][:, 1])
    f1 = 2 * (p * r) / (p + r + 1e-10)
    best_thresh = thresholds[np.argmax(f1)]
    y_pred_opt[:, i] = (probs[i][:, 1] >= best_thresh).astype(int)

/Users/frankiepike/ds_env/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


In [25]:
print("\nFinal Specialist Report:")
print(classification_report(y_test_spec, y_pred_opt, target_names=all_techs, zero_division=0))


Final Specialist Report:
                                    precision    recall  f1-score   support

               Appeal_to_Authority       0.38      0.80      0.52       234
          Appeal_to_fear-prejudice       0.34      0.61      0.43       188
    Bandwagon_Reductio_ad_hitlerum       0.66      0.81      0.73       426
           Black-and-White_Fallacy       0.14      0.32      0.20        57
         Causal_Oversimplification       0.16      0.53      0.25        74
                             Doubt       0.32      0.59      0.42       194
         Exaggeration_Minimisation       0.24      0.60      0.34       127
                       Flag-Waving       0.41      0.81      0.55       288
                   Loaded_Language       0.42      0.48      0.45       141
                      Minimisation       0.00      0.00      0.00         0
             Name_Calling_Labeling       0.34      0.35      0.34       129
                       Red_Herring       0.07      1.00      